In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl
import math
from typing import Dict, List, Optional

sys.path.append('../')

# corrected imports based on new project structure
from data.data_loading import load_all_raw_data
from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)
from analysis.data_analysis import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)
from analysis.station_analytics import analyze_station_activity, compare_stations
from data.weather_processing import WeatherDataCollector
from features.feature_engineering import (
    apply_all_enrichments,
    create_station_metadata_enhanced,
    add_coordinates_to_features,
    normalize_coordinates,
    add_fourier_coordinates,
    correct_trip_duration_mean,
    create_occurrence_flags,
    create_station_embeddings_mapping,
    add_station_embedding_index
)


In [2]:
# load base data with weather information
trips_with_weather = pl.read_parquet('../../data/processed/trips_with_weather.parquet')

print(f"loaded {trips_with_weather.shape[0]:,} trips with weather data")
print(f"columns: {len(trips_with_weather.columns)}")
print(f"date range: {trips_with_weather['fecha_origen_recorrido'].min()} to {trips_with_weather['fecha_origen_recorrido'].max()}")

trips_with_weather.head()


loaded 12,976,053 trips with weather data
columns: 48
date range: 2020-01-01 00:00:08 to 2024-08-30 23:57:59


id_recorrido,duracion_recorrido,fecha_origen_recorrido,id_estacion_origen,nombre_estacion_origen,direccion_estacion_origen,long_estacion_origen,lat_estacion_origen,fecha_destino_recorrido,id_estacion_destino,nombre_estacion_destino,direccion_estacion_destino,long_estacion_destino,lat_estacion_destino,id_usuario,modelo_bicicleta,genero,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_wind_speed_10m,weather_wind_speed_100m,weather_wind_direction_10m,weather_wind_direction_100m,weather_wind_gusts_10m,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation
i64,i64,datetime[ns],i64,str,str,f64,f64,datetime[ns],i64,str,str,f64,f64,i64,str,cat,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
7210548,1582,2020-01-24 21:54:39,27,"""027 - Montevideo""","""Cordoba Av. & Montevideo""",-58.390089,-34.599068,2020-01-24 22:21:01,3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,192009,"""ICONIC""","""FEMALE""",30.311,53.307747,19.761,31.59533,0.0,0.0,1.0,1006.8,1004.6493,45.0,0.0,7.0,43.0,0.2730162,2.015435,16.808569,23.400002,350.13425,345.74994,33.839996,33.060997,27.761,23.811,20.211,0.179,0.354,0.401,0.452,3600.0,1.0,190.0
7199093,204,2020-01-24 07:04:19,151,"""151 - AIME PAINÉ""","""Villaflor, Azucena & Paine, Ai…",-58.361285,-34.611815,2020-01-24 07:07:43,3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,36380,"""ICONIC""","""MALE""",24.211,70.07739,18.411001,25.07353,0.0,0.0,2.0,1011.9,1009.6941,50.0,0.0,0.0,50.0,0.060814,0.903983,15.328561,27.193705,350.53775,353.15732,28.08,24.361,26.661001,23.761,20.211,0.197,0.36,0.401,0.452,0.0,0.0,0.0
7196805,1790,2020-01-24 00:15:17,111,"""111 - MACACHA GUEMES""","""Machaca Guemes 350""",-58.364686,-34.605488,2020-01-24 00:45:07,3,"""003 - ADUANA""","""Moreno & Av Paseo Colon""",-58.36826,-34.611032,460080,"""ICONIC""","""MALE""",24.761,79.67906,21.011,26.88475,0.0,0.0,0.0,1012.2,1009.9975,0.0,0.0,0.0,0.0,0.038332,0.6342735,15.496736,23.177402,87.33706,83.75819,25.199999,29.311,27.361,23.661001,20.161001,0.197,0.361,0.401,0.452,0.0,0.0,0.0
7203598,10688,2020-01-24 12:38:16,285,"""400 - Reserva Ecologica""","""Achaval Rodriguez, T., Dr. Av.…",-58.356175,-34.617212,2020-01-24 15:36:24,4,"""004 - Plaza Roma""","""Lavalle & Bouchard""",-58.368781,-34.601822,3857,"""ICONIC""","""MALE""",26.011,62.75473,18.361,26.85348,0.0,0.0,3.0,1012.5,1010.3061,91.0,0.0,0.0,91.0,0.43236,1.2523198,18.11841,22.59699,339.04413,337.5205,37.44,26.311,25.711,23.761,20.211,0.198,0.361,0.401,0.452,3600.0,1.0,498.0
7200335,673,2020-01-24 08:31:01,171,"""171 - Pasteur""","""519 Pasteur""",-58.399755,-34.603281,2020-01-24 08:42:14,7,"""007 - OBELISCO""","""CARLOS PELEGRINI 215""",-58.381098,-34.606498,391034,"""ICONIC""","""FEMALE""",23.161001,77.50442,19.011,24.020298,0.0,0.0,0.0,1012.1,1009.8857,0.0,0.0,0.0,0.0,0.04618,0.6380112,17.283749,28.08923,1.1934711,1.4687687,28.44,23.711,26.361,23.761,20.211,0.197,0.36,0.401,0.452,0.0,0.0,0.0


In [3]:

trips_with_weather.null_count()


id_recorrido,duracion_recorrido,fecha_origen_recorrido,id_estacion_origen,nombre_estacion_origen,direccion_estacion_origen,long_estacion_origen,lat_estacion_origen,fecha_destino_recorrido,id_estacion_destino,nombre_estacion_destino,direccion_estacion_destino,long_estacion_destino,lat_estacion_destino,id_usuario,modelo_bicicleta,genero,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_wind_speed_10m,weather_wind_speed_100m,weather_wind_direction_10m,weather_wind_direction_100m,weather_wind_gusts_10m,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,43,43,43,43,43,0,0,71774,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
from features.core_pipeline import engineer_ecobici_features

df_feat = engineer_ecobici_features(trips_with_weather)

df_feat.describe()

🚴‍♂️  Parametrized Feature Engineering Pipeline (11 steps)
  Checkpoints directory: 'feature_checkpoints/'
  Input raw data shape: (12976053, 48)
  -> Memory usage after initialization: 398.0 MB
[Step 1/11] Normalizing Polars data...
  -> Loading Polars normalization from checkpoint: step01_polars_normalization.parquet
[Step 2/11] Creating departures table with lags & user profiles...


  -> Saved departures table checkpoint: step02_departures.parquet
[Step 3/11] Creating arrivals table with lags...


  -> Saved arrivals table checkpoint: step03_arrivals.parquet
[Step 4/11] Creating shifted arrivals table for target...


  -> Saved shifted arrivals checkpoint: step04_shifted_arrivals.parquet
[Step 5/11] Creating enhanced weather table...


  -> Saved weather table checkpoint: step05_weather.parquet
[Step 6/11] Creating station metadata table...


  -> Saved station metadata checkpoint: step06_stations.parquet
[Step 7/11] Building full station x time skeleton...


  -> Saved skeleton checkpoint: step07_skeleton.parquet
  -> Memory usage after after clearing main dataframe: 1042.3 MB
[Step 8/11] Joining data onto skeleton...


  -> Saved joined features checkpoint: step08_joined_with_weather.parquet
[Step 9/11] Engineering temporal and calendar features...
  -> Creating temporal features with memory optimization...
  -> Processing window: 2020-01-01 00:00:00 to 2020-01-08 00:00:00
  -> Memory usage after processed window until 2020-01-08 00:00:00: 985.0 MB
  -> Processing window: 2020-01-08 00:00:00 to 2020-01-15 00:00:00
  -> Memory usage after processed window until 2020-01-15 00:00:00: 1213.8 MB
  -> Processing window: 2020-01-15 00:00:00 to 2020-01-22 00:00:00
  -> Memory usage after processed window until 2020-01-22 00:00:00: 1207.6 MB
  -> Processing window: 2020-01-22 00:00:00 to 2020-01-29 00:00:00
  -> Memory usage after processed window until 2020-01-29 00:00:00: 1213.6 MB
  -> Processing window: 2020-01-29 00:00:00 to 2020-02-05 00:00:00
  -> Memory usage after processed window until 2020-02-05 00:00:00: 1233.9 MB
  -> Processing window: 2020-02-05 00:00:00 to 2020-02-12 00:00:00
  -> Memory usage

: 

In [6]:
df_feat = pl.read_parquet(r"../../data/processed/trips_feat_eng.parquet")



In [ ]:
# Obtener solo las columnas que tienen nulls (> 0)
df_nulls = df_feat.null_count()

null_summary = df_nulls.transpose(include_header=True)
null_summary.columns = ["column", "null_count"]

cols_with_nulls = null_summary.filter(pl.col("null_count") > 0)
print(cols_with_nulls)


shape: (9, 2)
┌───────────────────────┬────────────┐
│ column                ┆ null_count │
│ ---                   ┆ ---        │
│ str                   ┆ u32        │
╞═══════════════════════╪════════════╡
│ trip_dur_mean_last_DT ┆ 8448894    │
│ share_male            ┆ 8448894    │
│ share_female          ┆ 8448894    │
│ share_other           ┆ 8448894    │
│ precip_flag           ┆ 22860      │
│ wind_dir_sin          ┆ 22860      │
│ wind_dir_cos          ┆ 22860      │
│ precipitation_lag_1h  ┆ 23241      │
│ rain_lag_1h           ┆ 23241      │
└───────────────────────┴────────────┘


In [ ]:
df_feat.head()

station_id,ts_start,dep_last_DT,trip_dur_mean_last_DT,model_FIT_cnt,model_ICONIC_cnt,share_male,share_female,share_other,dep_lag_1,dep_lag_2,dep_lag_3,dep_lag_4,dep_lag_5,dep_lag_6,arr_last_DT,arr_lag_1,arr_lag_2,arr_lag_3,arr_lag_4,arr_lag_5,arr_lag_6,y_arrivals_next_DT,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_weather_code,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,…,weather_soil_temperature_0_to_7cm,weather_soil_temperature_7_to_28cm,weather_soil_temperature_28_to_100cm,weather_soil_temperature_100_to_255cm,weather_soil_moisture_0_to_7cm,weather_soil_moisture_7_to_28cm,weather_soil_moisture_28_to_100cm,weather_soil_moisture_100_to_255cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation,precip_flag,wind_dir_sin,wind_dir_cos,weather_code_cat_Clear,weather_code_cat_Clouds,weather_code_cat_Drizzle,weather_code_cat_Rain,weather_code_cat_null,precipitation_lag_1h,rain_lag_1h,is_holiday_ar,sin_hour,cos_hour,sin_dow,cos_dow,sin_month,cos_month,is_weekend,payday_flag,vacation_season,peak_commute,dep_ma_24h,dep_std_24h,dep_ratio_DT_24h,near_dep_sum_DT,near_dep_lag_1
i64,datetime[μs],u32,f64,u32,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,u8,u8,u8,u8,u8,f64,f64,i8,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,f32,f32,f32,u32,u32
2,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0
3,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,1,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0
4,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,1,0
5,2023-01-01 00:00:00,2,872.5,2,0,0.5,0.0,0.5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,2.0,0.0,0.666667,0,0
6,2023-01-01 00:00:00,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,22.261,69.92862,16.511,19.90884,0.0,0.0,3.0,1012.3,1010.0787,100.0,0.0,0.0,96.0,0.124625,…,27.361,27.611,23.561,19.511,0.157,0.321,0.426,0.43,0.0,0.0,0.0,0,0.931577,-0.363544,0,1,0,0,0,null,null,1,0.0,1.0,-2.4493e-16,1.0,0.5,0.866025,0,1,1,0,0.0,0.0,0.0,0,0


In [30]:
df_feat.columns

['station_id',
 'ts_start',
 'dep_last_DT',
 'trip_dur_mean_last_DT',
 'model_FIT_cnt',
 'model_ICONIC_cnt',
 'share_male',
 'share_female',
 'share_other',
 'dep_lag_1',
 'dep_lag_2',
 'dep_lag_3',
 'dep_lag_4',
 'dep_lag_5',
 'dep_lag_6',
 'arr_last_DT',
 'arr_lag_1',
 'arr_lag_2',
 'arr_lag_3',
 'arr_lag_4',
 'arr_lag_5',
 'arr_lag_6',
 'y_arrivals_next_DT',
 'weather_temperature_2m',
 'weather_relative_humidity_2m',
 'weather_dew_point_2m',
 'weather_apparent_temperature',
 'weather_precipitation',
 'weather_rain',
 'weather_weather_code',
 'weather_pressure_msl',
 'weather_surface_pressure',
 'weather_cloud_cover',
 'weather_cloud_cover_low',
 'weather_cloud_cover_mid',
 'weather_cloud_cover_high',
 'weather_et0_fao_evapotranspiration',
 'weather_vapour_pressure_deficit',
 'weather_wind_speed_10m',
 'weather_wind_speed_100m',
 'weather_wind_direction_10m',
 'weather_wind_direction_100m',
 'weather_wind_gusts_10m',
 'weather_soil_temperature_0_to_7cm',
 'weather_soil_temperature_7_

In [31]:
# # analizar una estación específica
# results = analyze_station_activity(df_feat, station_id=2, show_plots=True)

# # comparar múltiples estaciones
# compare_stations(df_feat, station_ids=[2, 27, 3, 111], metric='dep_last_DT')

In [9]:
code_cols = ["ts_start"]

target_col = 'y_arrivals_next_DT'

feat_cols = ['station_id', 'dep_last_DT', 'trip_dur_mean_last_DT',
       'model_FIT_cnt', 'model_ICONIC_cnt', 'share_male', 'share_female',
       'share_other', 'dep_lag_1', 'dep_lag_2', 'dep_lag_3', 'dep_lag_4',
       'dep_lag_5', 'dep_lag_6', 'arr_last_DT', 'arr_lag_1', 'arr_lag_2',
       'arr_lag_3', 'arr_lag_4', 'arr_lag_5', 'arr_lag_6', 'weather_temperature_2m',
       'weather_relative_humidity_2m', 'weather_dew_point_2m',
       'weather_apparent_temperature', 'weather_precipitation', 'weather_rain',
       'weather_pressure_msl', 'weather_surface_pressure', 'weather_cloud_cover',
       'weather_cloud_cover_low', 'weather_cloud_cover_mid',
       'weather_cloud_cover_high', 'weather_et0_fao_evapotranspiration',
       'weather_vapour_pressure_deficit',
       'weather_soil_temperature_0_to_7cm',
       'weather_soil_moisture_0_to_7cm', 'weather_sunshine_duration',
       'weather_is_day', 'weather_direct_radiation', 'precip_flag',
       'wind_dir_sin', 'wind_dir_cos', 'weather_code_cat_Clear',
       'weather_code_cat_Clouds', 'weather_code_cat_Drizzle',
       'weather_code_cat_Rain', 'weather_code_cat_null',
       'precipitation_lag_1h', 'rain_lag_1h', 'is_holiday_ar', 'sin_hour',
       'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month',
       'is_weekend', 'payday_flag', 'vacation_season', 'peak_commute',
       'dep_ma_24h', 'dep_std_24h', 'dep_ratio_DT_24h', 'near_dep_sum_DT',
       'near_dep_lag_1']



In [10]:
df_feat = df_feat[code_cols + feat_cols + [target_col]]

In [ ]:
# Apply data enrichments to the feature dataframe
print("🔧 APPLYING DATA ENRICHMENTS")
print("=" * 60)

# Load the raw data for coordinate extraction
print("Loading raw data for coordinate information...")
trips_with_weather = pl.read_parquet('../../data/processed/trips_with_weather.parquet')

# Apply all enrichments
df_feat_enriched, station_id_to_idx, embedding_dim = apply_all_enrichments(
    df=df_feat,
    df_raw=trips_with_weather,
    add_coords=True,
    add_fourier=True,
    k_values=[1, 2]
)

print(f"\n✅ Enrichment complete!")
print(f"   Original shape: {df_feat.shape}")
print(f"   Enriched shape: {df_feat_enriched.shape}")
print(f"   New columns added: {len(df_feat_enriched.columns) - len(df_feat.columns)}")

# Show new columns
new_cols = [col for col in df_feat_enriched.columns if col not in df_feat.columns]
print(f"   New columns: {new_cols}")

# Save embedding mapping for future use
import pickle
with open('../../data/processed/station_id_to_idx.pkl', 'wb') as f:
    pickle.dump(station_id_to_idx, f)
    
with open('../../data/processed/embedding_info.pkl', 'wb') as f:
    pickle.dump({'embedding_dim': embedding_dim, 'n_stations': len(station_id_to_idx)}, f)

print(f"\n💾 Saved embedding mapping ({len(station_id_to_idx)} stations, dim={embedding_dim})")

# Update df_feat to the enriched version
df_feat = df_feat_enriched.clone()

# Check for any remaining nulls in new columns
print("\n🔍 Checking for nulls in new columns:")
null_counts = df_feat.select([pl.col(col).null_count().alias(col) for col in new_cols]).to_dicts()[0]
for col, count in null_counts.items():
    if count > 0:
        print(f"   {col}: {count:,} nulls")
    else:
        print(f"   {col}: ✅ no nulls")


In [ ]:

# Show sample of enriched data
print("Sample of enriched data:")
enriched_sample = df_feat.select([
    "station_id", "station_id_idx", "ts_start",
    "lat", "lon", "lat_z", "lon_z", 
    "lat_sin_1", "lat_cos_1", "lon_sin_1", "lon_cos_1",
    "trip_dur_mean_last_DT", "dur_is_null", 
    "has_dep", "has_arr", "dep_last_DT", "y_arrivals_next_DT"
]).head(10)

display(enriched_sample)

# Statistics on new flags
print(f"\n📈 Flag Statistics:")
print(f"  dur_is_null: {df_feat['dur_is_null'].sum():,} / {df_feat.height:,} ({df_feat['dur_is_null'].mean()*100:.1f}%)")
print(f"  has_dep: {df_feat['has_dep'].sum():,} / {df_feat.height:,} ({df_feat['has_dep'].mean()*100:.1f}%)")  
print(f"  has_arr: {df_feat['has_arr'].sum():,} / {df_feat.height:,} ({df_feat['has_arr'].mean()*100:.1f}%)")

# Coordinate statistics
print(f"\n🗺️  Coordinate Statistics:")
coord_stats = df_feat.select([
    pl.col("lat").min().alias("lat_min"),
    pl.col("lat").max().alias("lat_max"), 
    pl.col("lat").mean().alias("lat_mean"),
    pl.col("lon").min().alias("lon_min"),
    pl.col("lon").max().alias("lon_max"),
    pl.col("lon").mean().alias("lon_mean")
]).to_dicts()[0]

for key, val in coord_stats.items():
    print(f"  {key}: {val:.6f}")

# Station embedding info
print(f"\n🏢 Station Embedding Info:")
print(f"  Total stations: {len(station_id_to_idx)}")
print(f"  Embedding dimension: {embedding_dim}")
print(f"  Station ID range: {min(station_id_to_idx.keys())} - {max(station_id_to_idx.keys())}")
print(f"  Index range: 0 - {max(station_id_to_idx.values())}")

# Check trip duration correction  
print(f"\n⏱️ Trip Duration Correction Results:")
duration_stats = df_feat.group_by("dur_is_null").agg([
    pl.col("trip_dur_mean_last_DT").mean().alias("avg_duration"),
    pl.col("trip_dur_mean_last_DT").count().alias("count"),
    pl.col("dep_last_DT").mean().alias("avg_departures")
])
display(duration_stats)


QUEDA DROPPEAR ALGUNAS COLUMNAS INNECESARIAS DE WEATHER Y ESTAMOS!

In [9]:
df_feat.sample(5)

#count number of uniques ids 

df_feat.select(pl.col("station_id").unique().count())

station_id
u32
381


In [ ]:

# Save the enriched feature dataframe
enriched_path = '../../data/processed/trips_feat_eng_enriched.parquet'
df_feat.write_parquet(enriched_path)
print(f"✅ Saved enriched features to: {enriched_path}")
print(f"   Shape: {df_feat.shape}")
print(f"   Size: {os.path.getsize(enriched_path) / 1024 / 1024:.1f} MB")

# Create a summary of all enrichments applied
enrichment_summary = {
    'original_columns': len([col for col in df_feat.columns if col not in new_cols]),
    'new_columns': len(new_cols), 
    'total_columns': len(df_feat.columns),
    'total_rows': df_feat.height,
    'stations_with_embeddings': len(station_id_to_idx),
    'embedding_dimension': embedding_dim,
    'has_coordinates': 'lat' in df_feat.columns and 'lon' in df_feat.columns,
    'has_fourier_features': any('sin_' in col or 'cos_' in col for col in df_feat.columns),
    'duration_corrections': df_feat['dur_is_null'].sum(),
    'occurrence_flags_added': 'has_dep' in df_feat.columns and 'has_arr' in df_feat.columns
}

print(f"\n📋 ENRICHMENT SUMMARY:")
for key, value in enrichment_summary.items():
    print(f"   {key}: {value}")

print(f"\n🎯 DATA READY FOR MODELING!")
print("   ✅ Entity embeddings (station_id_idx)")
print("   ✅ Spatial features (lat/lon + fourier)")  
print("   ✅ Clean duration features")
print("   ✅ Binary occurrence flags")
print("   ✅ All original features preserved")


In [ ]:

print("The enriched dataset now includes:")
print("✅ station_id_idx: 0-based indices for embedding lookup")
print("✅ Mapping saved to: ../../data/processed/station_id_to_idx.pkl") 
print("✅ Embedding info saved to: ../../data/processed/embedding_info.pkl")

print(f"\n📊 Embedding Configuration:")
print(f"   Number of stations: {len(station_id_to_idx)}")
print(f"   Recommended embedding dimension: {embedding_dim}")
print(f"   Total embedding parameters: {len(station_id_to_idx) * embedding_dim:,}")

print(f"\n🏗️ Example PyTorch Usage:")
print("""
import torch
import torch.nn as nn
from embedding_utils import StationEmbedding, EcoBiciModel

# Create station embedding layer
station_emb = StationEmbedding(n_stations=381, embedding_dim=19)

# Example forward pass
station_indices = torch.tensor([0, 1, 2, 10, 25])  # batch of station indices
embeddings = station_emb(station_indices)
print(f"Embeddings shape: {embeddings.shape}")  # [5, 19]

# Full model with embeddings
model = EcoBiciModel(n_stations=381, n_features=50, embedding_dim=19)
""")

print("📁 See src/embedding_utils.py for complete implementation!")

# Show example of station indices
print(f"\n🔍 Sample Station ID → Index Mapping:")
sample_stations = list(station_id_to_idx.items())[:10]
for station_id, idx in sample_stations:
    print(f"   Station {station_id} → Index {idx}")

print("\n✨ Your data is now ready for advanced neural network modeling!")
print("   Including entity embeddings, spatial features, and clean time series data.")
